The particle dynamics in a general velocity field $ \mathbf{u}(\mathbf{x}, t) = \begin{pmatrix} u(\mathbf{x}, t) \\ v(\mathbf{x}, t) \end{pmatrix} $. This function evaluates the velocity field $ \mathbf{u}(\mathbf{x}, t) $, at point $ \mathbf{x} $ at time $ t $.

| Name | Type (Shape) | Description |
| --- | --- | --- |
| t | float | time |
| x | array (2,) | $ \mathbf{x} $ |
| Interpolant_u | object | Interpolant object for $ u(\mathbf{x}, t) $ |
| Interpolant_v | object | Interpolant object for $ v(\mathbf{x}, t) $ |
| periodic | list (3,) | periodic[0]: periodicity in x <br /> periodic[1]: periodicity in y <br /> periodic[2]: periodicity in time|
| bool_unsteady | bool | specifies if velocity field is unsteady/steady |
| time_data | array(1,NT) | time of velocity data |
| vel | array (2,) | $ \mathbf{u}(\mathbf{x}, t) $ |

In [1]:
import sys, os

# get current directory
path = os.getcwd()

# get parent directory
parent_directory = os.path.sep.join(path.split(os.path.sep)[:-1])

# add utils folder to current working path in order to access the functions
sys.path.append(parent_directory+"/utils")

In [2]:
# Import numpy
import numpy as np

In [ ]:
import numpy as np

def velocity(t, x, X, Y,
    Interpolant_u, Interpolant_v,
    periodic, bool_unsteady, time_data,
    Interpolant_g=None,
    add_pro=False,
):
    """
    If add_pro=False (default): returns vel shape (2, N)
    If add_pro=True and Interpolant_g is provided: returns shape (3, N) with [u; v; g]
    """

    x_eval = x.copy()

    # periodic wrapping in x,y
    if periodic[0]:
        x_eval[0, :] = (x[0, :] - X[0, 0]) % (X[0, -1] - X[0, 0]) + X[0, 0]
    if periodic[1]:
        x_eval[1, :] = (x[1, :] - Y[0, 0]) % (Y[-1, 0] - Y[0, 0]) + Y[0, 0]

    # periodic wrapping in time
    if periodic[2]:
        t = t % (time_data[0, -1] - time_data[0, 0]) + time_data[0, 0]

    dt_data = time_data[0, 1] - time_data[0, 0]

    # helper: evaluate an unsteady list-of-interpolants at time t (linear interp in time)
    def eval_unsteady(interp_list):
        k = int((t - time_data[0, 0]) / dt_data)

        if k >= len(interp_list) - 1:
            return interp_list[-1](x_eval[1, :], x_eval[0, :], grid=False)

        Fi = interp_list[k](x_eval[1, :], x_eval[0, :], grid=False)
        Ff = interp_list[k + 1](x_eval[1, :], x_eval[0, :], grid=False)

        alpha = (t - (time_data[0, 0] + k * dt_data)) / dt_data
        return (1.0 - alpha) * Fi + alpha * Ff

    # -------- velocity (u,v) ----------
    if bool_unsteady:
        u = eval_unsteady(Interpolant_u)
        v = eval_unsteady(Interpolant_v)
    else:
        # steady case: Interpolant_u and Interpolant_v are callables (not lists)
        u = Interpolant_u(x_eval[1, :], x_eval[0, :], grid=False)
        v = Interpolant_v(x_eval[1, :], x_eval[0, :], grid=False)

    # -------- optional extra property g ----------
    if add_pro:
        if Interpolant_g is None:
            raise ValueError("add_pro=True but Interpolant_g was not provided.")

        if bool_unsteady:
            g = eval_unsteady(Interpolant_g)   # Interpolant_g should be a list like u/v
        else:
            g = Interpolant_g(x_eval[1, :], x_eval[0, :], grid=False)

        return np.vstack((u, v, g))  # (3, N)

    return np.vstack((u, v))        # (2, N)
